In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

In [2]:
INPUT_DIR = "/data/Minerva_20260125"

# Get all HDF5 files
data_files = sorted(Path(INPUT_DIR).glob("events_*.h5"))
print(f"Found {len(data_files)} dataset files:")
for f in data_files:
    print(f"  - {f.name}")

Found 12 dataset files:
  - events_1A.h5
  - events_1B.h5
  - events_1C.h5
  - events_1D.h5
  - events_1E.h5
  - events_1F.h5
  - events_1G.h5
  - events_1L.h5
  - events_1M.h5
  - events_1N.h5
  - events_1O.h5
  - events_1P.h5


In [3]:
def load_dataset_info(filepath):
    """Load basic information about a dataset."""
    with h5py.File(filepath, 'r') as f:
        info = {
            'n_events': f.attrs.get('n_events', len(f['data'])),
            'max_objects': f.attrs.get('max_objects', f['data'].shape[1]),
            'n_features': f.attrs.get('n_features', f['data'].shape[2]),
            'n_global_features': f.attrs.get('n_global_features', f['global'].shape[1] if 'global' in f else 0),
            'feature_names': f.attrs.get('feature_names', ['delta_eta', 'delta_phi', 'log_pt', 'log_E', 'pid']),
            'global_feature_names': f.attrs.get('global_feature_names', []),
        }
    return info

# Load info for all datasets
dataset_info = {}
for filepath in data_files:
    playlist = filepath.stem.replace('events_', '')
    dataset_info[playlist] = load_dataset_info(filepath)
    
# Print summary
print("Dataset Summary:")
print("=" * 80)
for playlist, info in dataset_info.items():
    print(f"{playlist}: {info['n_events']:,} events, {info['max_objects']} max objects, {info['n_features']} features per object")
print("=" * 80)
print(f"Total events: {sum(info['n_events'] for info in dataset_info.values()):,}")

Dataset Summary:
1A: 7,601,356 events, 100 max objects, 5 features per object
1B: 2,051,008 events, 100 max objects, 5 features per object
1C: 3,915,203 events, 100 max objects, 5 features per object
1D: 11,357,582 events, 100 max objects, 5 features per object
1E: 9,815,728 events, 100 max objects, 5 features per object
1F: 12,906,018 events, 100 max objects, 5 features per object
1G: 10,925,494 events, 100 max objects, 5 features per object
1L: 1,057,352 events, 100 max objects, 5 features per object
1M: 16,342,453 events, 100 max objects, 5 features per object
1N: 9,343,378 events, 100 max objects, 5 features per object
1O: 2,767,558 events, 100 max objects, 5 features per object
1P: 3,649,272 events, 100 max objects, 5 features per object
Total events: 91,732,402


In [ ]:
def compute_particle_statistics(filepath, sample_fraction=1.0):
    """
    Compute statistics about particle-level features.
    
    Args:
        filepath: Path to HDF5 file
        sample_fraction: Fraction of events to sample (1.0 = all events)
    """
    with h5py.File(filepath, 'r') as f:
        n_events = len(f['data'])
        
        # Sample events if needed
        if sample_fraction < 1.0:
            n_sample = int(n_events * sample_fraction)
            indices = np.random.choice(n_events, n_sample, replace=False)
            indices = np.sort(indices)  # HDF5 requires sorted indices
            data = f['data'][indices]
        else:
            data = f['data'][:]
        
        # Extract features (N_events, max_objects, n_features)
        # Features are: [delta_eta, delta_phi, log_pt, log_E, pid]
        
        # Create mask for valid particles (non-zero energy or non-zero features)
        valid_mask = np.any(data != 0, axis=2)  # (N_events, max_objects)
        
        # Flatten valid particles
        valid_data = data[valid_mask]  # (N_valid_particles, n_features)
        
        stats = {
            'n_events_sampled': len(data),
            'n_particles_total': np.sum(valid_mask),
            'n_particles_per_event_mean': np.mean(np.sum(valid_mask, axis=1)),
            'n_particles_per_event_std': np.std(np.sum(valid_mask, axis=1)),
            'n_particles_per_event_median': np.median(np.sum(valid_mask, axis=1)),
            'n_particles_per_event_min': np.min(np.sum(valid_mask, axis=1)),
            'n_particles_per_event_max': np.max(np.sum(valid_mask, axis=1)),
        }
        
        if len(valid_data) > 0:
            # Continuous features: delta_eta, delta_phi, log_pt, log_E
            for i, name in enumerate(['delta_eta', 'delta_phi', 'log_pt', 'log_E']):
                feature_values = valid_data[:, i]
                stats[f'{name}_mean'] = np.mean(feature_values)
                stats[f'{name}_std'] = np.std(feature_values)
                stats[f'{name}_median'] = np.median(feature_values)
                stats[f'{name}_min'] = np.min(feature_values)
                stats[f'{name}_max'] = np.max(feature_values)
                stats[f'{name}_q25'] = np.percentile(feature_values, 25)
                stats[f'{name}_q75'] = np.percentile(feature_values, 75)
            
            # Discrete feature: PID
            pid_values = valid_data[:, 4].astype(int)
            unique_pids, counts = np.unique(pid_values, return_counts=True)
            stats['unique_pids'] = unique_pids
            stats['pid_counts'] = counts
            stats['pid_fractions'] = counts / len(pid_values)
        
        return stats

# Compute statistics for all datasets
print("Computing particle statistics for all datasets...")
all_stats = {}
for filepath in data_files:
    playlist = filepath.stem.replace('events_', '')
    print(f"  Processing {playlist}...")
    all_stats[playlist] = compute_particle_statistics(filepath, sample_fraction=1.0)

print("✓ Statistics computed!")

Computing particle statistics for all datasets...
  Processing 1A...


/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:236: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


  Processing 1B...
  Processing 1C...


KeyboardInterrupt: 

In [ ]:
def compute_global_statistics(filepath, sample_fraction=1.0):
    """
    Compute statistics about event-level (global) features.
    """
    with h5py.File(filepath, 'r') as f:
        n_events = len(f['global'])
        
        # Sample events if needed
        if sample_fraction < 1.0:
            n_sample = int(n_events * sample_fraction)
            indices = np.random.choice(n_events, n_sample, replace=False)
            indices = np.sort(indices)  # HDF5 requires sorted indices
            global_data = f['global'][indices]
        else:
            global_data = f['global'][:]
        
        # Global features: [muon_fuzz_energy, muon_iso_blobs_energy, E_recoil, E_recoil_CCinc]
        # These are in log scale
        feature_names = ['muon_fuzz_energy', 'muon_iso_blobs_energy', 'E_recoil', 'E_recoil_CCinc']
        
        stats = {'n_events_sampled': len(global_data)}
        
        for i, name in enumerate(feature_names):
            feature_values = global_data[:, i]
            stats[f'{name}_mean'] = np.mean(feature_values)
            stats[f'{name}_std'] = np.std(feature_values)
            stats[f'{name}_median'] = np.median(feature_values)
            stats[f'{name}_min'] = np.min(feature_values)
            stats[f'{name}_max'] = np.max(feature_values)
            stats[f'{name}_q25'] = np.percentile(feature_values, 25)
            stats[f'{name}_q75'] = np.percentile(feature_values, 75)
        
        return stats

# Compute global statistics for all datasets
print("Computing global feature statistics for all datasets...")
global_stats = {}
for filepath in data_files:
    playlist = filepath.stem.replace('events_', '')
    print(f"  Processing {playlist}...")
    global_stats[playlist] = compute_global_statistics(filepath, sample_fraction=0.05)

print("✓ Global statistics computed!")

In [ ]:
def compute_truth_label_statistics(filepath):
    """
    Compute statistics about truth labels (incoming E and event type).
    """
    with h5py.File(filepath, 'r') as f:
        truth_labels = f['truth_labels'][:]
        
        # truth_labels shape: (N_events, 2)
        # Column 0: incoming neutrino energy (E_nu)
        # Column 1: event type (mc_intType)
        
        E_nu = truth_labels[:, 0]
        event_type = truth_labels[:, 1].astype(int)
        
        stats = {
            'n_events': len(truth_labels),
            # Energy statistics
            'E_nu_mean': np.mean(E_nu),
            'E_nu_std': np.std(E_nu),
            'E_nu_median': np.median(E_nu),
            'E_nu_min': np.min(E_nu),
            'E_nu_max': np.max(E_nu),
            'E_nu_q25': np.percentile(E_nu, 25),
            'E_nu_q75': np.percentile(E_nu, 75),
        }
        
        # Event type distribution
        unique_types, counts = np.unique(event_type, return_counts=True)
        stats['event_types'] = unique_types
        stats['event_type_counts'] = counts
        stats['event_type_fractions'] = counts / len(event_type)
        
        # Map to human-readable names
        type_names = {
            1: "CCQE",
            2: "Resonance",
            3: "DIS",
            4: "Coherent",
            8: "2p2h/MEC",
            10: "2p2h/MEC"
        }
        stats['event_type_names'] = [type_names.get(t, f"Other({t})") for t in unique_types]
        
        return stats

# Compute truth label statistics
print("Computing truth label statistics for all datasets...")
truth_stats = {}
for filepath in data_files:
    playlist = filepath.stem.replace('events_', '')
    print(f"  Processing {playlist}...")
    truth_stats[playlist] = compute_truth_label_statistics(filepath)

print("✓ Truth label statistics computed!")

# Summary Statistics Tables

## 1. Dataset Overview

In [ ]:
# Create overview table
overview_data = []
for playlist in sorted(dataset_info.keys()):
    info = dataset_info[playlist]
    particle_stats = all_stats[playlist]
    overview_data.append({
        'Playlist': playlist,
        'N Events': info['n_events'],
        'Avg Particles/Event': f"{particle_stats['n_particles_per_event_mean']:.1f}",
        'Median Particles/Event': f"{particle_stats['n_particles_per_event_median']:.0f}",
        'Max Particles/Event': f"{particle_stats['n_particles_per_event_max']:.0f}",
    })

overview_df = pd.DataFrame(overview_data)
print(overview_df.to_string(index=False))

# Total row
print("\n" + "="*80)
print(f"TOTAL: {overview_df['N Events'].sum():,} events across {len(overview_df)} playlists")

## 2. Particle-Level Feature Statistics (Continuous Variables)

In [ ]:
# Aggregate particle feature statistics across all playlists
feature_names = ['delta_eta', 'delta_phi', 'log_pt', 'log_E']

print("Particle Feature Statistics (aggregated across all playlists):")
print("=" * 100)

for feature in feature_names:
    # Weighted average across playlists
    total_particles = sum(stats['n_particles_total'] for stats in all_stats.values())
    
    weighted_mean = sum(stats[f'{feature}_mean'] * stats['n_particles_total'] 
                        for stats in all_stats.values()) / total_particles
    weighted_std = np.sqrt(sum(stats[f'{feature}_std']**2 * stats['n_particles_total'] 
                               for stats in all_stats.values()) / total_particles)
    
    # Overall min/max
    overall_min = min(stats[f'{feature}_min'] for stats in all_stats.values())
    overall_max = max(stats[f'{feature}_max'] for stats in all_stats.values())
    
    # Median of medians (approximate)
    median_of_medians = np.median([stats[f'{feature}_median'] for stats in all_stats.values()])
    
    print(f"\n{feature}:")
    print(f"  Mean: {weighted_mean:.4f} ± {weighted_std:.4f}")
    print(f"  Median: ~{median_of_medians:.4f}")
    print(f"  Range: [{overall_min:.4f}, {overall_max:.4f}]")

print("\n" + "=" * 100)

## 3. Particle Type (PID) Distribution

In [ ]:
# Aggregate PID distribution across all playlists
from collections import defaultdict

pid_total_counts = defaultdict(int)
total_particles = 0

for stats in all_stats.values():
    for pid, count in zip(stats['unique_pids'], stats['pid_counts']):
        pid_total_counts[int(pid)] += count
    total_particles += stats['n_particles_total']

# Sort by PID
sorted_pids = sorted(pid_total_counts.keys())

# Map PID to names
pid_names = {
    0: "Muon",
    1: "Photon",
    2: "Blob (no PID)",
    3: "Prong (PID=-999)",
    4: "Prong (PID=0)",
    5: "Prong (PID=3)",
    6: "Prong (PID=4)"
}

print("Particle Type Distribution (aggregated across all playlists):")
print("=" * 80)
print(f"{'PID':<5} {'Name':<25} {'Count':>15} {'Fraction':>12}")
print("-" * 80)

for pid in sorted_pids:
    count = pid_total_counts[pid]
    fraction = count / total_particles
    name = pid_names.get(pid, f"Unknown ({pid})")
    print(f"{pid:<5} {name:<25} {count:>15,} {fraction:>11.1%}")

print("-" * 80)
print(f"{'TOTAL':<5} {'':<25} {total_particles:>15,} {1.0:>11.1%}")
print("=" * 80)

## 4. Event Type (Interaction Type) Distribution

In [ ]:
# Aggregate event type distribution
event_type_counts = defaultdict(int)
total_events = 0

for stats in truth_stats.values():
    for evt_type, count in zip(stats['event_types'], stats['event_type_counts']):
        event_type_counts[int(evt_type)] += count
    total_events += stats['n_events']

# Sort by event type
sorted_types = sorted(event_type_counts.keys())

# Type names
type_names = {
    1: "CCQE",
    2: "Resonance/Delta",
    3: "DIS",
    4: "Coherent",
    8: "2p2h/MEC",
    10: "2p2h/MEC (alt)"
}

print("Event Type (Interaction Type) Distribution:")
print("=" * 80)
print(f"{'Type':<5} {'Name':<25} {'Count':>15} {'Fraction':>12}")
print("-" * 80)

for evt_type in sorted_types:
    count = event_type_counts[evt_type]
    fraction = count / total_events
    name = type_names.get(evt_type, f"Other ({evt_type})")
    print(f"{evt_type:<5} {name:<25} {count:>15,} {fraction:>11.1%}")

print("-" * 80)
print(f"{'TOTAL':<5} {'':<25} {total_events:>15,} {1.0:>11.1%}")
print("=" * 80)

## 5. Neutrino Energy (E_nu) Distribution

In [ ]:
# Aggregate E_nu statistics
total_events = sum(stats['n_events'] for stats in truth_stats.values())

weighted_E_nu_mean = sum(stats['E_nu_mean'] * stats['n_events'] 
                          for stats in truth_stats.values()) / total_events
weighted_E_nu_std = np.sqrt(sum(stats['E_nu_std']**2 * stats['n_events'] 
                                 for stats in truth_stats.values()) / total_events)

overall_E_nu_min = min(stats['E_nu_min'] for stats in truth_stats.values())
overall_E_nu_max = max(stats['E_nu_max'] for stats in truth_stats.values())
E_nu_median_of_medians = np.median([stats['E_nu_median'] for stats in truth_stats.values()])

print("Neutrino Energy (E_nu) Statistics:")
print("=" * 80)
print(f"Mean:     {weighted_E_nu_mean:.2f} MeV ± {weighted_E_nu_std:.2f} MeV")
print(f"          ({weighted_E_nu_mean/1000:.2f} GeV ± {weighted_E_nu_std/1000:.2f} GeV)")
print(f"Median:   ~{E_nu_median_of_medians:.2f} MeV (~{E_nu_median_of_medians/1000:.2f} GeV)")
print(f"Range:    [{overall_E_nu_min:.2f}, {overall_E_nu_max:.2f}] MeV")
print(f"          [{overall_E_nu_min/1000:.2f}, {overall_E_nu_max/1000:.2f}] GeV")
print("=" * 80)

## 6. Global Feature Statistics

In [ ]:
# Global feature statistics (note: these are in log scale)
global_feature_names = ['muon_fuzz_energy', 'muon_iso_blobs_energy', 'E_recoil', 'E_recoil_CCinc']

print("Global Feature Statistics (note: stored as log values):")
print("=" * 100)

total_events = sum(stats['n_events_sampled'] for stats in global_stats.values())

for feature in global_feature_names:
    weighted_mean = sum(stats[f'{feature}_mean'] * stats['n_events_sampled'] 
                        for stats in global_stats.values()) / total_events
    weighted_std = np.sqrt(sum(stats[f'{feature}_std']**2 * stats['n_events_sampled'] 
                               for stats in global_stats.values()) / total_events)
    
    overall_min = min(stats[f'{feature}_min'] for stats in global_stats.values())
    overall_max = max(stats[f'{feature}_max'] for stats in global_stats.values())
    median_of_medians = np.median([stats[f'{feature}_median'] for stats in global_stats.values()])
    
    print(f"\n{feature} (log scale):")
    print(f"  Mean: {weighted_mean:.4f} ± {weighted_std:.4f}")
    print(f"  Median: ~{median_of_medians:.4f}")
    print(f"  Range: [{overall_min:.4f}, {overall_max:.4f}]")

print("\n" + "=" * 100)

# Visualizations

In [ ]:
# Plot 1: Number of events per playlist
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

playlists = sorted(dataset_info.keys())
n_events = [dataset_info[p]['n_events'] for p in playlists]

ax.bar(playlists, n_events, color='steelblue', alpha=0.7)
ax.set_xlabel('Playlist', fontsize=12)
ax.set_ylabel('Number of Events', fontsize=12)
ax.set_title('Number of Events per Playlist', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (playlist, count) in enumerate(zip(playlists, n_events)):
    ax.text(i, count + max(n_events)*0.01, f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Particle type distribution (pie chart)
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

sorted_pids = sorted(pid_total_counts.keys())
labels = [pid_names.get(pid, f"PID {pid}") for pid in sorted_pids]
sizes = [pid_total_counts[pid] for pid in sorted_pids]
colors = plt.cm.Set3(np.linspace(0, 1, len(sorted_pids)))

wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                    colors=colors, startangle=90)

# Make percentage text more readable
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

ax.set_title('Particle Type Distribution (All Playlists)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Event type distribution (bar chart)
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

sorted_types = sorted(event_type_counts.keys())
type_labels = [type_names.get(t, f"Type {t}") for t in sorted_types]
type_counts = [event_type_counts[t] for t in sorted_types]
type_fractions = [c / total_events * 100 for c in type_counts]

colors_map = {
    1: 'blue',      # CCQE
    2: 'red',       # Resonance
    3: 'green',     # DIS
    4: 'purple',    # Coherent
    8: 'orange',    # 2p2h
    10: 'darkorange' # 2p2h alt
}
bar_colors = [colors_map.get(t, 'gray') for t in sorted_types]

bars = ax.bar(type_labels, type_counts, color=bar_colors, alpha=0.7)
ax.set_xlabel('Interaction Type', fontsize=12)
ax.set_ylabel('Number of Events', fontsize=12)
ax.set_title('Event Type (Interaction Type) Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

# Add percentage labels on bars
for i, (count, frac) in enumerate(zip(type_counts, type_fractions)):
    ax.text(i, count + max(type_counts)*0.01, f'{count:,}\n({frac:.1f}%)', 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Particle feature distributions (sample from first playlist)
# Load sample data for visualization
sample_playlist = sorted(data_files)[0]
print(f"Loading sample data from {sample_playlist.name} for visualization...")

with h5py.File(sample_playlist, 'r') as f:
    # Sample 10000 events
    n_sample = min(10000, len(f['data']))
    indices = np.random.choice(len(f['data']), n_sample, replace=False)
    sample_data = f['data'][indices]

# Extract valid particles
valid_mask = np.any(sample_data != 0, axis=2)
valid_particles = sample_data[valid_mask]

# Plot distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

feature_names_plot = ['delta_eta', 'delta_phi', 'log_pt', 'log_E']
feature_indices = [0, 1, 2, 3]

for i, (name, idx) in enumerate(zip(feature_names_plot, feature_indices)):
    ax = axes[i]
    values = valid_particles[:, idx]
    
    ax.hist(values, bins=100, alpha=0.7, color='steelblue', edgecolor='black')
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'Distribution of {name}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add statistics text
    mean_val = np.mean(values)
    std_val = np.std(values)
    ax.text(0.02, 0.98, f'μ={mean_val:.2f}\nσ={std_val:.2f}',
            transform=ax.transAxes, va='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Plot 5: Neutrino energy distribution
with h5py.File(sample_playlist, 'r') as f:
    n_sample = min(20000, len(f['truth_labels']))
    indices = np.random.choice(len(f['truth_labels']), n_sample, replace=False)
    truth_sample = f['truth_labels'][indices]

E_nu_sample = truth_sample[:, 0] / 1000  # Convert to GeV

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(E_nu_sample, bins=100, alpha=0.7, color='green', edgecolor='black')
axes[0].set_xlabel('Neutrino Energy (GeV)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Neutrino Energy Distribution (Linear Scale)', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add statistics
mean_E = np.mean(E_nu_sample)
median_E = np.median(E_nu_sample)
axes[0].axvline(mean_E, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_E:.2f} GeV')
axes[0].axvline(median_E, color='orange', linestyle='--', linewidth=2, label=f'Median = {median_E:.2f} GeV')
axes[0].legend()

# Log scale
axes[1].hist(E_nu_sample, bins=100, alpha=0.7, color='green', edgecolor='black')
axes[1].set_xlabel('Neutrino Energy (GeV)', fontsize=11)
axes[1].set_ylabel('Count (log scale)', fontsize=11)
axes[1].set_title('Neutrino Energy Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axvline(mean_E, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_E:.2f} GeV')
axes[1].axvline(median_E, color='orange', linestyle='--', linewidth=2, label=f'Median = {median_E:.2f} GeV')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot 6: Particles per event distribution
particles_per_event = []
for filepath in data_files:
    with h5py.File(filepath, 'r') as f:
        data = f['data'][:]
        valid_mask = np.any(data != 0, axis=2)
        particles_per_event.extend(np.sum(valid_mask, axis=1))

particles_per_event = np.array(particles_per_event)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(particles_per_event, bins=50, alpha=0.7, color='purple', edgecolor='black')
axes[0].set_xlabel('Number of Particles per Event', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Particle Multiplicity Distribution (Linear Scale)', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

mean_mult = np.mean(particles_per_event)
median_mult = np.median(particles_per_event)
axes[0].axvline(mean_mult, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_mult:.1f}')
axes[0].axvline(median_mult, color='orange', linestyle='--', linewidth=2, label=f'Median = {median_mult:.0f}')
axes[0].legend()

# Log scale
axes[1].hist(particles_per_event, bins=50, alpha=0.7, color='purple', edgecolor='black')
axes[1].set_xlabel('Number of Particles per Event', fontsize=11)
axes[1].set_ylabel('Count (log scale)', fontsize=11)
axes[1].set_title('Particle Multiplicity Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axvline(mean_mult, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_mult:.1f}')
axes[1].axvline(median_mult, color='orange', linestyle='--', linewidth=2, label=f'Median = {median_mult:.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Multiplicity statistics:")
print(f"  Mean: {mean_mult:.2f} particles/event")
print(f"  Median: {median_mult:.0f} particles/event")
print(f"  Std: {np.std(particles_per_event):.2f}")
print(f"  Min: {np.min(particles_per_event)}")
print(f"  Max: {np.max(particles_per_event)}")

# Summary Report

Save key statistics to a text file for reference.

In [ ]:
# Generate summary report
output_path = Path(INPUT_DIR) / "dataset_statistics_summary.txt"

with open(output_path, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write("MINERvA PREPROCESSED DATASET STATISTICS SUMMARY\n")
    f.write("=" * 100 + "\n\n")
    
    # Dataset overview
    f.write("DATASET OVERVIEW\n")
    f.write("-" * 100 + "\n")
    f.write(f"Number of playlists: {len(dataset_info)}\n")
    f.write(f"Total events: {sum(info['n_events'] for info in dataset_info.values()):,}\n")
    f.write(f"Total particles: {sum(stats['n_particles_total'] for stats in all_stats.values()):,}\n")
    f.write("\n")
    
    # Per-playlist breakdown
    f.write("PER-PLAYLIST BREAKDOWN\n")
    f.write("-" * 100 + "\n")
    for playlist in sorted(dataset_info.keys()):
        info = dataset_info[playlist]
        stats = all_stats[playlist]
        f.write(f"{playlist}: {info['n_events']:>8,} events, "
                f"{stats['n_particles_per_event_mean']:>5.1f} avg particles/event\n")
    f.write("\n")
    
    # Particle type distribution
    f.write("PARTICLE TYPE DISTRIBUTION\n")
    f.write("-" * 100 + "\n")
    for pid in sorted(pid_total_counts.keys()):
        count = pid_total_counts[pid]
        fraction = count / total_particles
        name = pid_names.get(pid, f"Unknown ({pid})")
        f.write(f"{name:<25}: {count:>12,} ({fraction:>6.1%})\n")
    f.write("\n")
    
    # Event type distribution
    f.write("EVENT TYPE (INTERACTION TYPE) DISTRIBUTION\n")
    f.write("-" * 100 + "\n")
    for evt_type in sorted(event_type_counts.keys()):
        count = event_type_counts[evt_type]
        fraction = count / total_events
        name = type_names.get(evt_type, f"Other ({evt_type})")
        f.write(f"{name:<25}: {count:>12,} ({fraction:>6.1%})\n")
    f.write("\n")
    
    # Neutrino energy
    f.write("NEUTRINO ENERGY STATISTICS\n")
    f.write("-" * 100 + "\n")
    f.write(f"Mean: {weighted_E_nu_mean:.2f} MeV ({weighted_E_nu_mean/1000:.2f} GeV)\n")
    f.write(f"Std:  {weighted_E_nu_std:.2f} MeV ({weighted_E_nu_std/1000:.2f} GeV)\n")
    f.write(f"Range: [{overall_E_nu_min:.2f}, {overall_E_nu_max:.2f}] MeV\n")
    f.write(f"       [{overall_E_nu_min/1000:.2f}, {overall_E_nu_max/1000:.2f}] GeV\n")
    f.write("\n")
    
    f.write("=" * 100 + "\n")

print(f"✓ Summary report saved to: {output_path}")
print(f"\nYou can view it with: cat {output_path}")